In [6]:
%pip install -q pandas scikit-learn shap lime matplotlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [10]:
from __future__ import annotations

import os
import sys
import zipfile
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
DATA_FILE = Path("german_credit.csv")
OUT_DIR = Path("rsa_practical_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Official Statlog German Credit (20 input attributes + class).
# The UCI archive contains german.data with whitespace-separated rows.
UCI_ZIP_URL = (
    "https://archive.ics.uci.edu/static/public/144/"
    "statlog+german+credit+data.zip"
)

COLUMN_NAMES = [
    "checking_status", "duration", "credit_history", "purpose",
    "credit_amount", "savings_status", "employment", "installment_rate",
    "personal_status", "other_parties", "residence_since", "property_magnitude",
    "age", "other_payment_plans", "housing", "existing_credits", "job",
    "num_dependents", "telephone", "foreign_worker", "class"
]

# These are categorical fields in the original German Credit data.
CATEGORICAL_COLUMNS = [
    "checking_status", "credit_history", "purpose", "savings_status",
    "employment", "personal_status", "other_parties", "property_magnitude",
    "other_payment_plans", "housing", "job", "telephone", "foreign_worker"
]


def create_csv_from_uci_if_needed(csv_path: Path) -> None:
    """Create german_credit.csv from the official UCI archive if it is absent."""
    if csv_path.exists():
        print(f"Using existing dataset: {csv_path.resolve()}")
        return

    print("german_credit.csv not found. Downloading the UCI Statlog German Credit dataset...")
    zip_path = OUT_DIR / "german_credit_uci.zip"
    try:
        urllib.request.urlretrieve(UCI_ZIP_URL, zip_path)
        with zipfile.ZipFile(zip_path, "r") as zf:
            # The archive typically contains german.data; locate it robustly.
            candidates = [
                name for name in zf.namelist()
                if Path(name).name.lower() == "german.data"
            ]
            if not candidates:
                raise FileNotFoundError(
                    "The UCI ZIP did not contain german.data. "
                    "Please upload german.data or german_credit.csv to Colab."
                )
            with zf.open(candidates[0], "r") as file:
                data = pd.read_csv(
                    file, sep=r"\s+", header=None, names=COLUMN_NAMES,
                    engine="python"
                )
        if data.shape != (1000, 21):
            raise ValueError(
                f"Unexpected dataset shape {data.shape}; expected 1000 rows and 21 columns."
            )
        data.to_csv(csv_path, index=False)
        print(f"Created {csv_path.resolve()} with shape {data.shape}")
    except Exception as exc:
        print("\nAutomatic download failed:", repr(exc))
        print("Please upload the official UCI german.data or a german_credit.csv "
              "containing the 20 attributes and class column.")
        raise


def load_dataset(csv_path: Path) -> pd.DataFrame:
    create_csv_from_uci_if_needed(csv_path)
    df = pd.read_csv(csv_path)

    # Allow common target-column spellings.
    target_candidates = ["class", "creditability", "target", "label", "risk"]
    target_col = next((c for c in target_candidates if c in df.columns), None)
    if target_col is None:
        raise ValueError(
            "Target column not found. Expected 'class' (UCI German Credit) "
            "or one of: creditability, target, label, risk."
        )
    if target_col != "class":
        df = df.rename(columns={target_col: "class"})

    # Ensure all 20 original attribute columns are present.
    missing = [c for c in COLUMN_NAMES[:-1] if c not in df.columns]
    if missing:
        raise ValueError(
            "Dataset is missing expected German Credit columns: "
            + ", ".join(missing)
            + ". Use the official Statlog German Credit dataset."
        )

    df = df[COLUMN_NAMES].copy()
    if df.isna().any().any():
        # The UCI original has no missing values. Fail clearly rather than
        # silently changing the practical dataset.
        missing_counts = df.isna().sum()
        missing_counts = missing_counts[missing_counts > 0]
        raise ValueError(f"Missing values found:\n{missing_counts}")

    # UCI class: 1 = good, 2 = bad. The practical requires bad as positive (1).
    values = set(pd.to_numeric(df["class"], errors="coerce").dropna().unique())
    if values.issubset({1, 2}) and values == {1, 2}:
        df["class"] = (pd.to_numeric(df["class"]) == 2).astype(int)
    else:
        # Also support already-labelled strings / binary 0,1 files.
        normalized = df["class"].astype(str).str.strip().str.lower()
        mapping = {
            "bad": 1, "good": 0, "1": 1, "0": 0,
            "2": 1, "good credit": 0, "bad credit": 1
        }
        if set(normalized.unique()).issubset(set(mapping)):
            # For the original numeric UCI 1/2 case, handled above.
            df["class"] = normalized.map(mapping).astype(int)
        else:
            raise ValueError(
                "Could not map target to bad=1 and good=0. "
                f"Observed target values: {sorted(normalized.unique())}"
            )

    # Make categorical columns strings for OrdinalEncoder.
    for col in CATEGORICAL_COLUMNS:
        df[col] = df[col].astype(str)

    print("\nDATASET")
    print(f"Rows: {len(df)} | Attributes: {len(COLUMN_NAMES)-1}")
    print("Target mapping: bad/rejected = 1, good/approved = 0")
    print("Class counts:", df["class"].value_counts().sort_index().to_dict())
    return df


def get_shap_values_for_bad_class(explainer, X):
    """Normalize SHAP output across common SHAP versions to class index 1."""
    values = explainer.shap_values(X)
    if isinstance(values, list):
        # Older SHAP versions: list[class_index] -> (rows, features)
        if len(values) > 1:
            return np.asarray(values[1])
        return np.asarray(values[0])

    values = np.asarray(values)
    # Newer SHAP versions for binary classifiers can return
    # (rows, features, classes).
    if values.ndim == 3:
        if values.shape[-1] > 1:
            return values[:, :, 1]
        return values[:, :, 0]
    if values.ndim == 2:
        return values
    raise ValueError(f"Unexpected SHAP values shape: {values.shape}")


def get_bad_class_base_value(explainer):
    base = explainer.expected_value
    if isinstance(base, (list, tuple, np.ndarray)):
        arr = np.asarray(base).reshape(-1)
        return float(arr[1] if len(arr) > 1 else arr[0])
    return float(base)


def main():
    print("RSA EXTERNAL PRACTICAL — GERMAN CREDIT EXPLAINABILITY")


    df = load_dataset(DATA_FILE)
    X_raw = df.drop(columns=["class"])
    y = df["class"].astype(int)

    # OrdinalEncoder: each original attribute remains exactly one feature.
    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )
    X_encoded = pd.DataFrame(
        encoder.fit_transform(X_raw),
        columns=X_raw.columns,
        index=X_raw.index
    )

    # Required 80:20 stratified split, random_state=42.
    X_train, X_test, y_train, y_test = train_test_split(
        X_encoded, y, test_size=0.20, stratify=y, random_state=SEED
    )
    print("\nTASK 1 — PREPROCESSING AND SPLIT")
    print(f"Train rows: {len(X_train)} | Test rows: {len(X_test)}")
    print(f"Encoded feature count: {X_encoded.shape[1]} (one per original attribute)")

    # Required Random Forest settings.
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        random_state=SEED,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    p_bad = model.predict_proba(X_test)[:, 1]
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, p_bad)

    print("\nTASK 2 — MODEL PERFORMANCE")
    print("Model: RandomForest (200 trees, depth 8)")
    print(f"Test accuracy: {accuracy:.4f}")
    print(f"ROC-AUC:       {auc:.4f}")

    # SHAP TreeExplainer. Use class 1 = bad/rejected.
    try:
        import shap
    except ImportError as exc:
        raise ImportError("SHAP is required. In Colab run: %pip install -q shap") from exc

    print("\nTASK 3 — GLOBAL SHAP EXPLANATION")
    shap_explainer = shap.TreeExplainer(model)
    shap_test_values = get_shap_values_for_bad_class(shap_explainer, X_test)
    mean_abs_shap = np.mean(np.abs(shap_test_values), axis=0)
    global_importance = pd.DataFrame({
        "feature": X_test.columns,
        "mean_abs_shap": mean_abs_shap
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    global_importance.insert(0, "rank", np.arange(1, len(global_importance) + 1))
    global_importance.to_csv(OUT_DIR / "shap_global_importance.csv", index=False)

    print("\nGLOBAL FEATURE IMPORTANCE (mean |SHAP|, class = bad)")
    print(global_importance.head(5).to_string(
        index=False, formatters={"mean_abs_shap": "{:.6f}".format}
    ))

    # SHAP summary (bar) is robust across SHAP versions and matches requested format.
    plt.figure(figsize=(10, 7))
    shap.summary_plot(
        shap_test_values, X_test, plot_type="bar",
        max_display=20, show=False
    )
    plt.title("Global SHAP Feature Importance — Class: Bad")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "shap_summary.png", dpi=180, bbox_inches="tight")
    plt.close()
    print(f"Plot saved: {OUT_DIR / 'shap_summary.png'}")

    # Select a test applicant predicted bad (positive class).
    bad_positions = np.flatnonzero(y_pred == 1)
    if len(bad_positions) == 0:
        # If no predicted bad rows, use the highest bad-probability test row.
        selected_pos = int(np.argmax(p_bad))
        print("\nWarning: Model predicted no test rows as bad; selecting the highest P(bad) row.")
    else:
        # Prefer the first predicted-bad row in test order, reproducible split.
        selected_pos = int(bad_positions[0])

    selected_index = X_test.index[selected_pos]
    applicant = X_test.iloc[selected_pos]
    applicant_shap = shap_test_values[selected_pos]
    applicant_probability = float(p_bad[selected_pos])
    predicted_bad = int(y_pred[selected_pos]) == 1

    print("\nTASK 4 — LOCAL SHAP EXPLANATION")
    print(f"Applicant (original row index {selected_index}): "
          f"P(bad) = {applicant_probability:.3f} -> "
          f"{'REJECTED' if predicted_bad else 'NOT REJECTED (fallback applicant)'}")

    local_shap = pd.DataFrame({
        "feature": X_test.columns,
        "encoded_value": applicant.values,
        "shap_value": applicant_shap
    })
    # Positive SHAP increases model output for bad class; negative decreases it.
    local_shap["direction"] = np.where(
        local_shap["shap_value"] >= 0,
        "increases bad/rejection score",
        "decreases bad/rejection score"
    )
    local_shap["abs_shap"] = np.abs(local_shap["shap_value"])
    local_shap = local_shap.sort_values("abs_shap", ascending=False).reset_index(drop=True)
    local_shap.to_csv(OUT_DIR / "selected_applicant_shap.csv", index=False)

    print("\nSHAP top 3 (direction is relative to class bad):")
    for _, row in local_shap.head(3).iterrows():
        print(f"  {row['feature']} = {row['encoded_value']:.3f} "
              f"{row['shap_value']:+.4f}")

    # Draw waterfall using modern Explanation API; fallback to a signed bar chart.
    try:
        base_value = get_bad_class_base_value(shap_explainer)
        explanation = shap.Explanation(
            values=applicant_shap,
            base_values=base_value,
            data=applicant.values,
            feature_names=list(X_test.columns)
        )
        plt.figure(figsize=(10, 7))
        shap.plots.waterfall(explanation, max_display=15, show=False)
        plt.tight_layout()
        plt.savefig(OUT_DIR / "shap_local_waterfall.png", dpi=180, bbox_inches="tight")
        plt.close()
    except Exception as exc:
        print("Waterfall rendering fallback used:", str(exc)[:200])
        top_local = local_shap.head(12).sort_values("shap_value")
        plt.figure(figsize=(10, 6))
        plt.barh(top_local["feature"], top_local["shap_value"])
        plt.axvline(0, linewidth=1)
        plt.xlabel("SHAP value for class bad")
        plt.title("Local SHAP Explanation (fallback bar plot)")
        plt.tight_layout()
        plt.savefig(OUT_DIR / "shap_local_waterfall.png", dpi=180, bbox_inches="tight")
        plt.close()
    print(f"Plot saved: {OUT_DIR / 'shap_local_waterfall.png'}")

    # LIME uses encoded features. Feature names are retained; categorical feature
    # indices are marked categorical so LIME can describe discrete categories.
    print("\nTASK 5 — LIME EXPLANATION")
    try:
        from lime.lime_tabular import LimeTabularExplainer
    except ImportError as exc:
        raise ImportError("LIME is required. In Colab run: %pip install -q lime") from exc

    categorical_indices = [
        X_train.columns.get_loc(col) for col in CATEGORICAL_COLUMNS
    ]
    lime_explainer = LimeTabularExplainer(
        training_data=X_train.to_numpy(),
        feature_names=list(X_train.columns),
        class_names=["good/approved", "bad/rejected"],
        categorical_features=categorical_indices,
        discretize_continuous=True,
        mode="classification",
        random_state=SEED
    )
    lime_exp = lime_explainer.explain_instance(
        data_row=applicant.to_numpy(),
        predict_fn=model.predict_proba,
        num_features=5,
        labels=(1,)
    )
    lime_list = lime_exp.as_list(label=1)
    lime_df = pd.DataFrame(lime_list, columns=["lime_rule", "lime_weight"])
    lime_df.to_csv(OUT_DIR / "selected_applicant_lime.csv", index=False)
    print("LIME top 5 (positive weight supports bad/rejection):")
    for rule, weight in lime_list[:5]:
        print(f"  {rule}: {weight:+.4f}")

    # Compare feature names in top 3. LIME rule text starts with a feature name;
    # map to known original feature names to avoid brittle string parsing.
    def lime_feature_from_rule(rule_text):
        matches = [name for name in X_test.columns if rule_text.startswith(name)]
        return max(matches, key=len) if matches else None

    lime_top3_features = [
        lime_feature_from_rule(rule) for rule, _ in lime_list[:3]
    ]
    shap_top3_features = local_shap.head(3)["feature"].tolist()
    shap_set = set(shap_top3_features)
    lime_set = {name for name in lime_top3_features if name is not None}
    agreement = len(shap_set.intersection(lime_set))

    print("\nSHAP top 3 features:", shap_top3_features)
    print("LIME top 3 features:", lime_top3_features)
    print(f"Top-3 agreement (feature overlap): {agreement} / 3")

    # Decode original category values for an applicant-facing explanation.
    # Encoder.categories_ is aligned with the original categorical columns.
    category_lookup = {
        col: {float(i): str(category) for i, category in enumerate(categories)}
        for col, categories in zip(CATEGORICAL_COLUMNS, encoder.categories_)
    }

    def original_value(feature):
        raw_value = float(applicant[feature])
        if feature in category_lookup:
            return category_lookup[feature].get(raw_value, str(raw_value))
        # Numeric attributes are integer-valued in this dataset.
        return str(int(round(raw_value)))

    # Identify whether any listed reason is a sensitive attribute.
    sensitive_attributes = {"age", "personal_status", "foreign_worker"}
    top3_reason_features = set(shap_top3_features)
    sensitive_in_reasons = sorted(top3_reason_features.intersection(sensitive_attributes))

    print("\nTASK 6 — SENSITIVE ATTRIBUTE CHECK")
    print("Sensitive attributes in SHAP top-3 reasons:",
          ", ".join(sensitive_in_reasons) if sensitive_in_reasons else "None")
    if sensitive_in_reasons:
        print("REVIEW: sensitive attribute(s) appear; investigate fairness and policy before use.")
    else:
        print("No listed sensitive attribute appears in the SHAP top-3 reasons. "
              "Still perform a broader fairness review.")

    # Plain-language message uses legitimate non-sensitive features only.
    # Checking status and duration are translated using applicant's actual categories.
    legitimate_reasons = [
        feature for feature in shap_top3_features
        if feature not in sensitive_attributes
    ][:2]
    if not legitimate_reasons:
        legitimate_reasons = [
            feature for feature in local_shap["feature"].tolist()
            if feature not in sensitive_attributes
        ][:2]

    reason_phrases = []
    for feature in legitimate_reasons:
        value = original_value(feature)
        if feature == "checking_status":
            phrase = f"your current account status ({value})"
        elif feature == "duration":
            phrase = f"the requested loan duration ({value} months)"
        elif feature == "credit_amount":
            phrase = f"the requested credit amount ({value})"
        elif feature == "savings_status":
            phrase = f"your savings status ({value})"
        elif feature == "purpose":
            phrase = f"the stated loan purpose ({value})"
        elif feature == "credit_history":
            phrase = f"your credit history ({value})"
        elif feature == "employment":
            phrase = f"your employment status ({value})"
        elif feature == "installment_rate":
            phrase = f"the installment rate ({value})"
        elif feature == "existing_credits":
            phrase = f"the number of existing credits ({value})"
        else:
            phrase = f"{feature.replace('_', ' ')} ({value})"
        reason_phrases.append(phrase)

    if len(reason_phrases) >= 2:
        rejection_message = (
            "Your EMI request was not approved mainly because of "
            + reason_phrases[0] + " and " + reason_phrases[1] + "."
        )
    elif reason_phrases:
        rejection_message = (
            "Your EMI request was not approved mainly because of "
            + reason_phrases[0] + "."
        )
    else:
        rejection_message = (
            "Your EMI request was not approved based on the model's overall "
            "assessment. Please contact the finance team for a human review."
        )

    print("\nREJECTION MESSAGE (legitimate reasons only):")
    print('"' + rejection_message + '"')

    # Save a full run summary to use in the practical report.
    with open(OUT_DIR / "results_summary.txt", "w", encoding="utf-8") as f:
        f.write("RSA EXTERNAL PRACTICAL — OBSERVED RESULTS\\n")
        f.write(f"Dataset: UCI Statlog German Credit (1000 applicants, 20 attributes)\\n")
        f.write("Target mapping: bad=1, good=0\\n")
        f.write("Split: 80:20 stratified, random_state=42\\n")
        f.write("Model: RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)\\n")
        f.write(f"Accuracy: {accuracy:.6f}\\nROC-AUC: {auc:.6f}\\n\\n")
        f.write("Top 5 global SHAP features:\\n")
        for _, row in global_importance.head(5).iterrows():
            f.write(f"{int(row['rank'])}. {row['feature']}: {row['mean_abs_shap']:.6f}\\n")
        f.write("\\nSelected applicant:\\n")
        f.write(f"Original row index: {selected_index}\\n")
        f.write(f"P(bad): {applicant_probability:.6f}\\n")
        f.write(f"Prediction: {'bad/rejected' if predicted_bad else 'good/not rejected'}\\n")
        f.write("Top 3 SHAP reasons (feature, value, SHAP):\\n")
        for _, row in local_shap.head(3).iterrows():
            f.write(f"- {row['feature']} = {original_value(row['feature'])}; "
                    f"SHAP={row['shap_value']:+.6f}\\n")
        f.write("\\nLIME top 5:\\n")
        for rule, weight in lime_list[:5]:
            f.write(f"- {rule}: {weight:+.6f}\\n")
        f.write(f"Top-3 feature agreement: {agreement}/3\\n")
        f.write("Sensitive attributes in SHAP top-3: "
                + (", ".join(sensitive_in_reasons) if sensitive_in_reasons else "None")
                + "\\n")
        f.write("Rejection message: " + rejection_message + "\\n")

    # Save reproducible model and preprocessing object.
    import joblib
    joblib.dump(
        {"model": model, "encoder": encoder, "feature_names": list(X_encoded.columns)},
        OUT_DIR / "random_forest_and_encoder.joblib"
    )

    print("\nOUTPUT FILES")
    for filename in [
        "shap_summary.png",
        "shap_local_waterfall.png",
        "shap_global_importance.csv",
        "selected_applicant_shap.csv",
        "selected_applicant_lime.csv",
        "results_summary.txt",
        "random_forest_and_encoder.joblib"
    ]:
        print(f"  {OUT_DIR / filename}")
    print("\nCompleted successfully.")



if __name__ == "__main__":
    main()


RSA EXTERNAL PRACTICAL — GERMAN CREDIT EXPLAINABILITY
Using existing dataset: /content/german_credit.csv

DATASET
Rows: 1000 | Attributes: 20
Target mapping: bad/rejected = 1, good/approved = 0
Class counts: {0: 700, 1: 300}

TASK 1 — PREPROCESSING AND SPLIT
Train rows: 800 | Test rows: 200
Encoded feature count: 20 (one per original attribute)

TASK 2 — MODEL PERFORMANCE
Model: RandomForest (200 trees, depth 8)
Test accuracy: 0.7700
ROC-AUC:       0.8121

TASK 3 — GLOBAL SHAP EXPLANATION

GLOBAL FEATURE IMPORTANCE (mean |SHAP|, class = bad)
 rank         feature mean_abs_shap
    1 checking_status      0.086602
    2        duration      0.041237
    3  savings_status      0.039146
    4  credit_history      0.025840
    5   credit_amount      0.025818
Plot saved: rsa_practical_output/shap_summary.png

TASK 4 — LOCAL SHAP EXPLANATION
Applicant (original row index 289): P(bad) = 0.533 -> REJECTED

SHAP top 3 (direction is relative to class bad):
  checking_status = 0.000 +0.1131
  dura